In [1]:
import cv2 as cv
import numpy as np
from keras.models import load_model
import os
from collections import deque
import glob

# Configuration
IMAGE_HEIGHT, IMAGE_WIDTH = 224, 224
SEQUENCE_LENGTH = 10
CLASSES_LIST = ["accident", "fighting", "fire", "normal_resized"]

# Color mappings for different classes
CLASS_COLORS = {
    "accident": (0, 0, 255),     # Red
    "fighting": (0, 0, 255),     # Red  
    "fire": (0, 0, 255),         # Red
    "normal_resized": (0, 255, 0) # Green
}

# Text mappings
CLASS_TEXTS = {
    "accident": "Abnormal [ Accident ]",
    "fighting": "Abnormal [ Fighting ]",
    "fire": "Abnormal [ Fire ]",
    "normal_resized": "Normal"
}

print("Configuration loaded successfully!")

Configuration loaded successfully!


In [2]:
# ========================================
# TEST BASIC PACKAGES
# ========================================
# Test if basic packages work before loading TensorFlow/Keras

print("Testing basic packages...")

try:
    import numpy as np
    print(f"✅ NumPy version: {np.__version__}")
except Exception as e:
    print(f"❌ NumPy failed: {e}")

try:
    import cv2 as cv
    print(f"✅ OpenCV version: {cv.__version__}")
except Exception as e:
    print(f"❌ OpenCV failed: {e}")

try:
    import os
    import glob
    from collections import deque
    print("✅ Standard library modules imported successfully")
except Exception as e:
    print(f"❌ Standard library modules failed: {e}")

print("\nBasic package test complete! If all packages loaded successfully, proceed to the next cell.")

Testing basic packages...
✅ NumPy version: 2.2.6
✅ OpenCV version: 4.12.0
✅ Standard library modules imported successfully

Basic package test complete! If all packages loaded successfully, proceed to the next cell.


In [3]:
def process_video(input_video_path, output_video_path, model):
    """
    Process a single video file and make predictions on each sequence
    """
    cap = cv.VideoCapture(input_video_path)
    
    if not cap.isOpened():
        print(f"❌ Could not open video: {input_video_path}")
        return False
    
    # Get video properties
    fps = int(cap.get(cv.CAP_PROP_FPS))
    width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
    
    print(f"Processing: {os.path.basename(input_video_path)}")
    print(f"  - Dimensions: {width}x{height}")
    print(f"  - FPS: {fps}")
    print(f"  - Total frames: {total_frames}")
    
    # Initialize video writer
    fourcc = cv.VideoWriter_fourcc(*'mp4v')
    out = cv.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    # Initialize frame queue and variables
    frames_queue = deque(maxlen=SEQUENCE_LENGTH)
    predicted_class_name = ""
    confidence = 0.0
    frame_count = 0
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        frame_count += 1
        
        # Preprocess frame for model
        resized_frame = cv.resize(frame, (IMAGE_HEIGHT, IMAGE_WIDTH))
        normalized_frame = resized_frame / 255.0
        frames_queue.append(normalized_frame)
        
        # Make prediction when we have enough frames
        if len(frames_queue) == SEQUENCE_LENGTH:
            input_frames = np.array(frames_queue)
            input_frames = np.expand_dims(input_frames, axis=0)
            
            predictions = model.predict(input_frames, verbose=0)
            predicted_label = np.argmax(predictions)
            confidence = predictions[0][predicted_label] * 100
            predicted_class_name = CLASSES_LIST[predicted_label]
        
        # Add prediction text to frame
        if predicted_class_name:
            class_text = CLASS_TEXTS[predicted_class_name]
            color = CLASS_COLORS[predicted_class_name]
            
            # Main prediction text
            text = f"{class_text} ({confidence:.1f}%)"
            
            # Add text with background for better visibility
            font = cv.FONT_HERSHEY_SIMPLEX
            font_scale = 0.8
            thickness = 2
            
            # Get text size for background rectangle
            (text_width, text_height), baseline = cv.getTextSize(text, font, font_scale, thickness)
            
            # Draw background rectangle
            cv.rectangle(frame, (10, 10), (10 + text_width + 10, 10 + text_height + baseline + 10), (0, 0, 0), -1)
            
            # Draw text
            cv.putText(frame, text, (15, 10 + text_height + 5), font, font_scale, color, thickness, cv.LINE_AA)
        
        # Write frame to output video
        out.write(frame)
        
        # Show progress
        if frame_count % 30 == 0:
            progress = (frame_count / total_frames) * 100
            print(f"  Progress: {progress:.1f}% - Current prediction: {predicted_class_name}")
    
    # Release everything
    cap.release()
    out.release()
    
    print(f"✅ Video processed successfully: {os.path.basename(output_video_path)}")
    return True

In [4]:
def load_model_safely(model_path):
    """
    Safely load the Keras model with multiple fallback methods
    """
    print("Attempting to load model...")
    
    # Try different import methods
    load_model_func = None
    
    # Method 1: Try TensorFlow's Keras
    try:
        from tensorflow.keras.models import load_model as tf_load_model
        load_model_func = tf_load_model
        print("✅ Using tensorflow.keras.models.load_model")
    except Exception as e:
        print(f"⚠️ tensorflow.keras import failed: {e}")
    
    # Method 2: Try standalone Keras
    if load_model_func is None:
        try:
            from keras.models import load_model as keras_load_model
            load_model_func = keras_load_model
            print("✅ Using keras.models.load_model")
        except Exception as e:
            print(f"⚠️ keras import failed: {e}")
    
    # Method 3: Try tf.keras directly
    if load_model_func is None:
        try:
            import tensorflow as tf
            load_model_func = tf.keras.models.load_model
            print("✅ Using tf.keras.models.load_model")
        except Exception as e:
            print(f"⚠️ tf.keras import failed: {e}")
    
    if load_model_func is None:
        raise ImportError("Could not import any Keras model loading function. Please check your TensorFlow/Keras installation.")
    
    # Load the model
    try:
        model = load_model_func(model_path)
        print("✅ Model loaded successfully!")
        return model
    except Exception as e:
        raise Exception(f"Failed to load model from {model_path}: {e}")

def process_all_videos(input_folder, output_folder, model_path):
    """
    Process all videos in the input folder and save results to output folder
    """
    # Load the model using our safe loading function
    model = load_model_safely(model_path)
    
    # Create output folder if it doesn't exist
    os.makedirs(output_folder, exist_ok=True)
    
    # Get all video files from input folder
    video_extensions = ['*.mp4', '*.avi', '*.mov', '*.mkv', '*.wmv', '*.flv', '*.webm']
    video_files = []
    
    for extension in video_extensions:
        video_files.extend(glob.glob(os.path.join(input_folder, extension)))
        video_files.extend(glob.glob(os.path.join(input_folder, extension.upper())))
    
    if not video_files:
        print(f"❌ No video files found in: {input_folder}")
        return
    
    print(f"Found {len(video_files)} video file(s) to process:")
    for video_file in video_files:
        print(f"  - {os.path.basename(video_file)}")
    
    print("\\n" + "="*50)
    
    # Process each video one by one
    successful_count = 0
    for i, video_file in enumerate(video_files, 1):
        print(f"\\nProcessing video {i}/{len(video_files)}")
        
        # Create output filename
        input_filename = os.path.basename(video_file)
        name_without_ext = os.path.splitext(input_filename)[0]
        output_filename = f"{name_without_ext}_predicted.mp4"
        output_path = os.path.join(output_folder, output_filename)
        
        # Process the video
        if process_video(video_file, output_path, model):
            successful_count += 1
        
        print("-" * 30)
    
    print(f"\\n🎉 Processing complete!")
    print(f"Successfully processed: {successful_count}/{len(video_files)} videos")
    print(f"Output videos saved to: {output_folder}")

In [ ]:
# ========================================
# CONFIGURATION - Modify these paths as needed
# ========================================

# Path to your trained model
MODEL_PATH = r"f:/Study/Academic/Part 4/Project/Sentinel/TestWithWebCam/final_cv_model_optimized.keras"

# Input folder containing test videos
INPUT_FOLDER = r"f:/Study/Academic/Part 4/Project/Sentinel/TestWithWebCam/inputs"

# Output folder for processed videos with predictions
OUTPUT_FOLDER = r"f:/Study/Academic/Part 4/Project/Sentinel/TestWithWebCam/test_output_videos"

print("Configuration:")
print(f"Model Path: {MODEL_PATH}")
print(f"Input Folder: {INPUT_FOLDER}")
print(f"Output Folder: {OUTPUT_FOLDER}")
print(f"\\nModel exists: {os.path.exists(MODEL_PATH)}")
print(f"Input folder exists: {os.path.exists(INPUT_FOLDER)}")

# Create input folder if it doesn't exist
if not os.path.exists(INPUT_FOLDER):
    os.makedirs(INPUT_FOLDER)
    print(f"\\n📁 Created input folder: {INPUT_FOLDER}")
    print("Please add your test videos to this folder and run the next cell.")

Configuration:
Model Path: f:/Study/Academic/Part 4/Project/Sentinel/TestWithWebCam/final_cv_model_optimized.keras
Input Folder: f:/Study/Academic/Part 4/Project/Sentinel/TestWithWebCam/
Output Folder: f:/Study/Academic/Part 4/Project/Sentinel/TestWithWebCam/
\nModel exists: True
Input folder exists: True


In [8]:
# ========================================
# RUN VIDEO PROCESSING
# ========================================

# Execute this cell to process all videos in the input folder
if __name__ == "__main__":
    try:
        # Check if model exists
        if not os.path.exists(MODEL_PATH):
            print(f"❌ Model not found at: {MODEL_PATH}")
            print("Please check the model path and try again.")
        else:
            # Start processing
            print("🚀 Starting video processing...")
            process_all_videos(INPUT_FOLDER, OUTPUT_FOLDER, MODEL_PATH)
            
    except Exception as e:
        print(f"❌ An error occurred: {str(e)}")
        import traceback
        traceback.print_exc()

🚀 Starting video processing...
Attempting to load model...
✅ Using tensorflow.keras.models.load_model
✅ Model loaded successfully!
Found 2 video file(s) to process:
  - Big Fight 😔🥹😡 #shorts #shortsfeed #funny #viralshorts #video.mp4
  - Big Fight 😔🥹😡 #shorts #shortsfeed #funny #viralshorts #video.mp4
\n==================================================
\nProcessing video 1/2
Processing: Big Fight 😔🥹😡 #shorts #shortsfeed #funny #viralshorts #video.mp4
  - Dimensions: 360x640
  - FPS: 30
  - Total frames: 496
✅ Model loaded successfully!
Found 2 video file(s) to process:
  - Big Fight 😔🥹😡 #shorts #shortsfeed #funny #viralshorts #video.mp4
  - Big Fight 😔🥹😡 #shorts #shortsfeed #funny #viralshorts #video.mp4
\n==================================================
\nProcessing video 1/2
Processing: Big Fight 😔🥹😡 #shorts #shortsfeed #funny #viralshorts #video.mp4
  - Dimensions: 360x640
  - FPS: 30
  - Total frames: 496
  Progress: 6.0% - Current prediction: accident
  Progress: 6.0% - Current

# Video Testing Notebook for Activity Detection

## 📖 Instructions

1. **Setup**: Run the first cell to import all required libraries and configure the settings.

2. **Configuration**: 
   - The model path is set to use `final_cv_model_optimized.keras`
   - Input videos should be placed in `test_input_videos` folder
   - Processed videos will be saved to `test_output_videos` folder

3. **Add Test Videos**: Place your test videos in the `test_input_videos` folder. Supported formats:
   - MP4, AVI, MOV, MKV, WMV, FLV, WebM

4. **Run Processing**: Execute the final cell to process all videos.

## 🎯 Features

- **Sequence-based Prediction**: Uses sequences of 10 frames for prediction
- **Visual Feedback**: Adds prediction text with confidence percentage on each frame
- **Color Coding**:
  - 🔴 **Red**: Abnormal activities (Accident, Fighting, Fire)
  - 🟢 **Green**: Normal activities
- **Batch Processing**: Processes all videos in the input folder one by one
- **Progress Tracking**: Shows processing progress for each video

## 📝 Output

Each processed video will have:
- Prediction text in the top-left corner
- Confidence percentage
- Bold font for better visibility
- Background rectangle for text clarity

---

**Ready to test your videos!** 🎬